# PyTorch 입문 — 사용자 정의 Dataset · DataLoader

데이터가 **내 파일 형식**일 때 `Dataset`을 직접 만들고, `DataLoader`로 **배치**를 꺼내는 연습입니다.  
이론 복습: [`../session04.md`](../session04.md)

> **이 노트북이 하는 일**  
> 학습 파이프라인 개요 → `hi_set` Dataset → 간단 `transform` → `DataLoader`로 shape 확인.

환경: `torch-env` 커널을 쓰는 것을 권장합니다.

## 1. 학습 파이프라인 개요

1. 데이터 불러오기  
2. 전처리(transform)  
3. train/valid 나누기(이번 노트북에서는 생략)  
4. `DataLoader`로 배치 만들기  
5. 배치 shape 확인

In [2]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

## 2. 사용자 정의 Dataset — hi_set

`__init__`에서 데이터를 준비하고, `__len__`은 개수, `__getitem__`은 **한 샘플**을 돌려줍니다.

In [3]:
class hi_set(Dataset):
    def __init__(self, length=10, transform=None):
        torch.manual_seed(101) # 101번 시드를 사용하여 랜덤 시드를 고정
        self.x = 10 * torch.ones(length, 2)
        # length x 2 크기의 1로 초기화된 행렬 생성. 10을 곱하여 행렬을 10으로 초기화  
        self.y = torch.ones(length, 1)
        # length x 1 크기의 1로 초기화된 행렬 생성

        self.transform = transform # transform 함수를 저장
        self.len = length # length를 저장

    def __len__(self):
        return self.len # length를 반환

    def __getitem__(self, idx):
        sample_data = self.x[idx], self.y[idx] # idx 위치의 x와 y를 반환

        if self.transform:
            # transform 함수가 있으면 transform 함수를 적용
            sample_data = self.transform(sample_data)
        return sample_data

## 3. transform — scaling (최소 전처리)

원본 노트북의 `scaling`은 정의되지 않았습니다. 여기서는 입력 `x`를 10으로 나누어 **0~10 스케일을 0~1**로 맞춥니다.

In [4]:
def scaling(sample):
    x, y = sample # sample의 x와 y를 분리
    return x / 10.0, y # x를 10으로 나누어 0~10 스케일을 0~1로 맞춤

## 4. Dataset 하나 꺼내 보기

In [5]:
# hi_set 클래스를 사용하여 dataset을 생성, scaling 함수를 적용
dataset = hi_set(length=10, transform=scaling) # length=10, transform=scaling 인자를 사용하여 dataset을 생성
x0, y0 = dataset[0] # dataset의 0번째 인덱스의 x0과 y0을 반환
# [0] 인 이유? __getitem__ 메서드에서 idx 0을 반환하기 때문
# length 10 -> dataset.x 에 10x2 행렬을 생성, dataset.y 에 10x1 행렬을 생성
print("샘플 0 x:", x0, "y:", y0) # 샘플 0 x: tensor([1., 1.]), y: tensor([1.])
print("데이터 개수:", len(dataset)) # 데이터 개수: 10

샘플 0 x: tensor([1., 1.]) y: tensor([1.])
데이터 개수: 10


## 5. DataLoader — 배치 shape 확인

`batch_size=4`이면 한 번에 4개 샘플이 묶입니다. `shuffle=True`는 순서를 섞습니다.

In [6]:
# batch_size=4, shuffle=True 인자를 사용하여 DataLoader를 생성
# batch_size 4 -> 한 번에 4개 샘플이 묶임
# shuffle=True -> 순서를 섞음
loader = DataLoader(dataset, batch_size=4, shuffle=True)
# dataset의 길이 10 -> loader의 길이 10/4 = 2.5 -> 3개의 배치 생성
# dataset x 10x2, y 10x1 -> batch_x 4x2 / 4x2 / 2x2, batch_y 4x1 / 4x1 / 2x1 로 변경

for batch_x, batch_y in loader: # loader를 순회하며 batch_x와 batch_y를 반환
    print("batch_x shape:", batch_x.shape)
    print("batch_y shape:", batch_y.shape)
    break

batch_x shape: torch.Size([4, 2])
batch_y shape: torch.Size([4, 1])


---

이론 복습: [`../session04.md`](../session04.md) · 연산 복습: [`03_tensor_matmul_ops.ipynb`](./03_tensor_matmul_ops.ipynb)